# Notebook 4: Signal Generation + Backtest

Generates trading signals from the z-score of the HYSYS-weighted crack spread
and backtests the strategy with realistic transaction costs and an
**operational inertia friction term**.

## The operational inertia friction term

A refinery cannot instantly change its yield slate — feed heaters, column pressures,
and draw configurations take time to adjust. A minimum holding period of **5 days**
is enforced between signal changes. This is derived directly from the HYSYS process
simulation and distinguishes this strategy from a generic statistical model.

## Signal logic
- **Long** (z-score < −1.5): spread too compressed, buy the margin
- **Short** (z-score > +1.5): spread too wide, sell the margin
- **Exit** (|z-score| < 0): revert to mean

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load spread data with z-scores
df = pd.read_csv('../data/spread_with_zscore.csv', index_col=0, parse_dates=True)
df = df.dropna(subset=['zscore'])
print(f'Loaded {len(df)} trading days for backtest')

In [ ]:
# ============================================================
# STRATEGY PARAMETERS
# ============================================================
ENTRY_THRESHOLD  = 1.5    # z-score to enter trade
EXIT_THRESHOLD   = 0.0    # z-score to exit trade
TRANSACTION_COST = 0.05   # USD/bbl per trade (round-trip = 2×)
MIN_HOLDING_DAYS = 5      # Operational inertia — derived from HYSYS process constraints

print(f'Entry threshold:      ±{ENTRY_THRESHOLD} σ')
print(f'Exit threshold:        {EXIT_THRESHOLD} σ')
print(f'Transaction cost:     ${TRANSACTION_COST}/bbl per trade')
print(f'Min holding period:    {MIN_HOLDING_DAYS} days (operational inertia)')

In [ ]:
# ============================================================
# SIGNAL GENERATION
# ============================================================

def generate_signals(zscore, entry, exit_thresh):
    signals = pd.Series(0, index=zscore.index)
    position = 0
    for i in range(len(zscore)):
        z = zscore.iloc[i]
        if position == 0:
            if z < -entry:
                position = 1   # Long
            elif z > entry:
                position = -1  # Short
        elif position == 1:
            if z >= exit_thresh:
                position = 0   # Exit long
        elif position == -1:
            if z <= -exit_thresh:
                position = 0   # Exit short
        signals.iloc[i] = position
    return signals

df['signal_raw'] = generate_signals(df['zscore'], ENTRY_THRESHOLD, EXIT_THRESHOLD)

print(f'Signal distribution:')
print(df['signal_raw'].value_counts())

In [ ]:
# ============================================================
# OPERATIONAL INERTIA FRICTION TERM
# A refinery cannot instantly switch yield slate.
# Minimum 5-day holding period between signal changes.
# Derived from HYSYS process constraints.
# ============================================================

def apply_holding_period(signals, min_hold):
    held = signals.copy()
    last_change_idx = -min_hold
    prev_signal = 0
    for i in range(len(signals)):
        if signals.iloc[i] != prev_signal:
            if (i - last_change_idx) >= min_hold:
                held.iloc[i] = signals.iloc[i]
                last_change_idx = i
                prev_signal = signals.iloc[i]
            else:
                held.iloc[i] = prev_signal
        else:
            held.iloc[i] = prev_signal
    return held

df['signal'] = apply_holding_period(df['signal_raw'], MIN_HOLDING_DAYS)

trades_filtered = (df['signal'].diff().abs() > 0).sum()
trades_raw      = (df['signal_raw'].diff().abs() > 0).sum()
print(f'Trades before holding period filter: {trades_raw}')
print(f'Trades after  holding period filter: {trades_filtered}')

In [ ]:
# ============================================================
# PnL CALCULATION
# ============================================================

df['margin_change'] = df['margin_hysys'].diff()
df['pnl_gross']     = df['signal'].shift(1) * df['margin_change']
df['trade_flag']    = df['signal'].diff().abs().clip(0, 1)
df['pnl_net']       = df['pnl_gross'] - df['trade_flag'] * TRANSACTION_COST
df['equity_curve']  = df['pnl_net'].cumsum()

print('Daily PnL summary ($/bbl):')
print(df['pnl_net'].describe().round(4))

In [ ]:
# ============================================================
# PERFORMANCE METRICS
# ============================================================

total_return  = df['pnl_net'].sum()
daily_mean    = df['pnl_net'].mean()
daily_std     = df['pnl_net'].std()
sharpe        = (daily_mean / daily_std) * np.sqrt(252) if daily_std > 0 else 0
max_drawdown  = (df['equity_curve'] - df['equity_curve'].cummax()).min()
win_rate      = (df['pnl_net'] > 0).mean()
num_trades    = df['trade_flag'].sum() / 2
avg_hold_days = len(df) / max(num_trades, 1)

print('=' * 45)
print('BACKTEST PERFORMANCE SUMMARY')
print('=' * 45)
print(f'Total Return:       ${total_return:.2f}/bbl')
print(f'Sharpe Ratio:        {sharpe:.2f}')
print(f'Max Drawdown:       ${max_drawdown:.2f}/bbl')
print(f'Win Rate:            {win_rate:.1%}')
print(f'Number of Trades:    {num_trades:.0f}')
print(f'Avg Holding Period:  {avg_hold_days:.0f} days')
print(f'Strategy:            Entry ±{ENTRY_THRESHOLD}σ, Min Hold {MIN_HOLDING_DAYS}d, TC ${TRANSACTION_COST}/bbl')

In [ ]:
# ============================================================
# PERFORMANCE PLOTS
# ============================================================

fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(3, 1, hspace=0.35)

# 1. Equity curve
ax1 = fig.add_subplot(gs[0])
ax1.plot(df.index, df['equity_curve'], color='#2c3e50', linewidth=1.5)
ax1.fill_between(df.index, df['equity_curve'], 0,
    where=(df['equity_curve'] >= 0), alpha=0.15, color='#27ae60')
ax1.fill_between(df.index, df['equity_curve'], 0,
    where=(df['equity_curve'] < 0),  alpha=0.15, color='#e74c3c')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax1.set_ylabel('Cumulative PnL ($/bbl)')
ax1.set_title(f'Equity Curve — Sharpe: {sharpe:.2f} | Max DD: ${max_drawdown:.2f}/bbl | Trades: {num_trades:.0f}')
ax1.grid(alpha=0.3)

# 2. Z-score with signals overlaid
ax2 = fig.add_subplot(gs[1])
ax2.plot(df.index, df['zscore'], color='#7f8c8d', linewidth=0.7, alpha=0.8)
ax2.axhline(0,    color='black',   linewidth=0.8)
ax2.axhline(1.5,  color='#e74c3c', linewidth=1, linestyle='--', alpha=0.7)
ax2.axhline(-1.5, color='#27ae60', linewidth=1, linestyle='--', alpha=0.7)
long_entries  = df[(df['signal'].diff() == 1)].index
short_entries = df[(df['signal'].diff() == -1)].index
exits         = df[(df['signal'].diff() != 0) & (df['signal'] == 0)].index
ax2.scatter(long_entries,  df.loc[long_entries,  'zscore'], marker='^', color='#27ae60', s=40, zorder=5, label='Long entry')
ax2.scatter(short_entries, df.loc[short_entries, 'zscore'], marker='v', color='#e74c3c', s=40, zorder=5, label='Short entry')
ax2.scatter(exits,         df.loc[exits,         'zscore'], marker='x', color='#2c3e50', s=30, zorder=5, label='Exit')
ax2.set_ylabel('Z-score')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# 3. Drawdown
drawdown = df['equity_curve'] - df['equity_curve'].cummax()
ax3 = fig.add_subplot(gs[2])
ax3.fill_between(df.index, drawdown, 0, alpha=0.5, color='#e74c3c')
ax3.set_ylabel('Drawdown ($/bbl)')
ax3.set_xlabel('Date')
ax3.set_title('Drawdown')
ax3.grid(alpha=0.3)

plt.savefig('../data/backtest_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# SENSITIVITY ANALYSIS — Entry threshold vs Sharpe
# ============================================================

thresholds = [0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5]
results = []

for thresh in thresholds:
    sig  = generate_signals(df['zscore'], thresh, 0.0)
    sig  = apply_holding_period(sig, MIN_HOLDING_DAYS)
    pnl  = sig.shift(1) * df['margin_change'] - sig.diff().abs().clip(0,1) * TRANSACTION_COST
    sh   = (pnl.mean() / pnl.std()) * np.sqrt(252) if pnl.std() > 0 else 0
    ntrd = sig.diff().abs().clip(0,1).sum() / 2
    results.append({'threshold': thresh, 'sharpe': sh, 'num_trades': ntrd, 'total_pnl': pnl.sum()})

sensitivity = pd.DataFrame(results)
print(sensitivity.round(3))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sensitivity['threshold'], sensitivity['sharpe'], 'o-', color='#2c3e50', linewidth=2)
ax.axvline(ENTRY_THRESHOLD, color='#e74c3c', linestyle='--', label=f'Selected threshold: {ENTRY_THRESHOLD}')
ax.set_xlabel('Entry Threshold (σ)')
ax.set_ylabel('Annualised Sharpe Ratio')
ax.set_title('Sensitivity Analysis — Entry Threshold vs Sharpe Ratio')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save final results
df[['margin_hysys', 'zscore', 'signal', 'pnl_gross', 'pnl_net', 'equity_curve']].to_csv('../data/backtest_results.csv')

summary = {
    'total_return_per_bbl': round(total_return, 2),
    'sharpe_ratio': round(sharpe, 2),
    'max_drawdown_per_bbl': round(max_drawdown, 2),
    'win_rate': round(win_rate, 3),
    'num_trades': int(num_trades),
    'avg_hold_days': round(avg_hold_days, 1),
    'entry_threshold_sigma': ENTRY_THRESHOLD,
    'min_holding_days': MIN_HOLDING_DAYS,
    'transaction_cost_per_bbl': TRANSACTION_COST
}

pd.Series(summary).to_csv('../data/performance_summary.csv', header=['value'])
print('Saved backtest results.')
print('\nFinal performance summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')